In [25]:
# EDA
import pandas as pd
import plotly.express as px
import seaborn as sns
import numpy as np

sns.set_style("whitegrid")

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Carregar Dados

In [26]:
# Carregar os dados já tratados
df_leads = pd.read_csv('datasets/leads_cleaned.csv')

In [27]:
# Mostrar as linhas iniciais
df_leads.head(20)

,Lead Origin,Lead Source,Do Not Email,Do Not Call,Converted,TotalVisits,Total Time Spent on Website,Page Views Per Visit,Last Activity,Search,Newspaper Article,X Education Forums,Newspaper,Digital Advertisement,Through Recommendations,A free copy of Mastering The Interview,Last Notable Activity
0,API,Olark Chat,0,0,0,0.0,0,0.00,Page Visited on Website,0,0,0,0,0,0,0,Modified
1,API,Organic Search,0,0,0,5.0,674,2.50,Email Opened,0,0,0,0,0,0,0,Email Opened
2,Landing Page Submission,Direct Traffic,0,0,1,2.0,1532,2.00,Email Opened,0,0,0,0,0,0,1,Email Opened
3,Landing Page Submission,Direct Traffic,0,0,0,1.0,305,1.00,Unreachable,0,0,0,0,0,0,0,Modified
4,Landing Page Submission,Google,0,0,1,2.0,1428,1.00,Converted to Lead,0,0,0,0,0,0,0,Modified
5,API,Olark Chat,0,0,0,0.0,0,0.00,Olark Chat Conversation,0,0,0,0,0,0,0,Modified
6,Landing Page Submission,Google,0,0,1,2.0,1640,2.00,Email Opened,0,0,0,0,0,0,0,Modified
7,API,Olark Chat,0,0,0,0.0,0,0.00,Olark Chat Conversation,0,0,0,0,0,0,0,Modified
8,Landing Page Submission,Direct Traffic,0,0,0,2.0,71,2.00,Email Opened,0,0,0,0,0,0,1,Email Opened
9,API,Google,0,0,0,4.0,58,4.00,Email Opened,0,0,0,0,0,0,0,Email Opened


In [28]:
# Mostrar as últimas linhas
df_leads.tail(20)

,Lead Origin,Lead Source,Do Not Email,Do Not Call,Converted,TotalVisits,Total Time Spent on Website,Page Views Per Visit,Last Activity,Search,Newspaper Article,X Education Forums,Newspaper,Digital Advertisement,Through Recommendations,A free copy of Mastering The Interview,Last Notable Activity
9054,Landing Page Submission,Direct Traffic,0,0,0,5.0,20,2.50,SMS Sent,0,0,0,0,0,0,1,Modified
9055,Landing Page Submission,Google,0,0,0,4.0,1347,2.00,SMS Sent,0,0,0,0,0,0,1,SMS Sent
9056,API,Google,0,0,0,6.0,228,6.00,SMS Sent,0,0,0,0,0,0,0,Modified
9057,API,Organic Search,0,0,0,7.0,142,7.00,Email Opened,0,0,0,0,0,0,1,Modified
9058,Landing Page Submission,Google,0,0,0,4.0,455,4.00,Form Submitted on Website,0,0,0,0,0,0,0,Modified
9059,Landing Page Submission,Direct Traffic,1,0,0,2.0,74,2.00,Email Bounced,0,0,0,0,0,0,1,Modified
9060,API,Olark Chat,0,0,0,0.0,0,0.00,SMS Sent,0,0,0,0,0,0,0,Modified
9061,Landing Page Submission,Google,0,0,1,5.0,1283,1.67,Email Opened,0,0,0,0,0,0,0,Email Opened
9062,Landing Page Submission,Google,0,0,1,4.0,1944,2.00,SMS Sent,0,0,0,0,0,0,1,Modified
9063,Landing Page Submission,Organic Search,0,0,1,13.0,1226,6.50,SMS Sent,0,0,0,0,0,0,1,Modified


In [29]:
# Estrutura do dataset
df_leads.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9074 entries, 0 to 9073
Data columns (total 17 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Lead Origin                             9074 non-null   object 
 1   Lead Source                             9074 non-null   object 
 2   Do Not Email                            9074 non-null   int64  
 3   Do Not Call                             9074 non-null   int64  
 4   Converted                               9074 non-null   int64  
 5   TotalVisits                             9074 non-null   float64
 6   Total Time Spent on Website             9074 non-null   int64  
 7   Page Views Per Visit                    9074 non-null   float64
 8   Last Activity                           9074 non-null   object 
 9   Search                                  9074 non-null   int64  
 10  Newspaper Article                       9074 non-null   int6

# Preparação dos dados

In [30]:
# Preparar os dados para o modelo
X = df_leads.drop(columns=['Converted'])
y = df_leads['Converted']

In [31]:
# Criar lista de colunas
numeric_features = X.select_dtypes(include=['number']).columns
categorical_features = X.select_dtypes(include='object').columns

In [32]:
# Usar preprocessor existente
import joblib
preprocessor = joblib.load('preprocessor_dataset_leads.pkl')

In [33]:
# Dividir o dataset entre treinamento e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=51)

In [34]:
# Aplicar o preprocessor
X_train = preprocessor.fit_transform(X_train).toarray()
X_test = preprocessor.transform(X_test).toarray()

In [35]:
# Mostrar os conjuntos
print(f'Conjunto de treinamento: {X_train.shape}')
print(f'Conjunto de testes: {X_test.shape}')

Conjunto de treinamento: (7259, 68)
Conjunto de testes: (1815, 68)


# Treinamento do modelo

In [36]:
# Criar o modelo de StackingClassifier

# Meta-Modelo
lr_model_vanilla = LogisticRegression(random_state=51)

# Modelos Base
tree_model_vanilla = DecisionTreeClassifier(random_state=51)
svc_model_vanilla = SVC(kernel='linear')
sgd_model_vanilla = SGDClassifier(penalty='elasticnet', random_state=51)

# Criar o objeto do StackingClassifier
stacking_model_vanilla = StackingClassifier(
    estimators=[
        ('sgd classifier', sgd_model_vanilla),
        ('svc', svc_model_vanilla),
        ('decision tree', tree_model_vanilla)
    ],
    final_estimator=lr_model_vanilla,
    # Passthrough = False, usa apenas os resultados dos estimadores de cada algoritmo base (Vanilla)
    # Passthrough = True, usa os resultados dos estimadores de cada algortimo base + dataset original (Blending)
    passthrough=False 
)

In [37]:
# Criar o modelo de StackingClassifier

# Meta-Modelo
lr_model_blending = LogisticRegression(random_state=51)

# Modelos Base
tree_model_blending = DecisionTreeClassifier(random_state=51)
svc_model_blending = SVC(kernel='linear')
sgd_model_blending = SGDClassifier(penalty='elasticnet', random_state=51)

# Criar o objeto do StackingClassifier
stacking_model_blending = StackingClassifier(
    estimators=[
        ('sgd classifier', sgd_model_blending),
        ('svc', svc_model_blending),
        ('decision tree', tree_model_blending)
    ],
    final_estimator=lr_model_blending,
    # Passthrough = False, usa apenas os resultados dos estimadores de cada algoritmo base (Vanilla)
    # Passthrough = True, usa os resultados dos estimadores de cada algortimo base + dataset original (Blending)
    passthrough=True 
)

In [38]:
# Treinar o modelo Vanilla
stacking_model_vanilla.fit(X_train, y_train)

StackingClassifier(estimators=[('sgd classifier',
                                SGDClassifier(penalty='elasticnet',
                                              random_state=51)),
                               ('svc', SVC(kernel='linear')),
                               ('decision tree',
                                DecisionTreeClassifier(random_state=51))],
                   final_estimator=LogisticRegression(random_state=51))

In [39]:
# Treinar o modelo Vanilla
stacking_model_blending.fit(X_train, y_train)

StackingClassifier(estimators=[('sgd classifier',
                                SGDClassifier(penalty='elasticnet',
                                              random_state=51)),
                               ('svc', SVC(kernel='linear')),
                               ('decision tree',
                                DecisionTreeClassifier(random_state=51))],
                   final_estimator=LogisticRegression(random_state=51),
                   passthrough=True)

# Avaliação do Modelo

In [40]:
# Fazer predições no conjunto de testes
y_pred_vanilla = stacking_model_vanilla.predict(X_test)

In [41]:
# Fazer predições no conjunto de testes
y_pred_blending = stacking_model_blending.predict(X_test)

In [42]:
# Calcular métricas vanilla
accuracy_vanilla = accuracy_score(y_test, y_pred_vanilla)
precision_vanilla = precision_score(y_test, y_pred_vanilla)
recall_vanilla = recall_score(y_test, y_pred_vanilla)
f1_vanilla = f1_score(y_test, y_pred_vanilla)

In [43]:
# Calcular métricas blending
accuracy_blending = accuracy_score(y_test, y_pred_blending)
precision_blending = precision_score(y_test, y_pred_blending)
recall_blending = recall_score(y_test, y_pred_blending)
f1_blending = f1_score(y_test, y_pred_blending)

In [44]:
# Apresentar as métricas vanilla
print("Métricas do modelo Vanilla:")
print(f'Acurácia: {accuracy_vanilla}')
print(f'Precisão: {precision_vanilla}')
print(f'Recall: {recall_vanilla}')
print(f'F1-Score: {f1_vanilla}')

Métricas do modelo Vanilla:
Acurácia: 0.8
Precisão: 0.74481658692185
Recall: 0.6970149253731344
F1-Score: 0.7201233616037008


In [45]:
# Apresentar as métricas blending
print("Métricas do modelo blending:")
print(f'Acurácia: {accuracy_blending}')
print(f'Precisão: {precision_blending}')
print(f'Recall: {recall_blending}')
print(f'F1-Score: {f1_blending}')

Métricas do modelo blending:
Acurácia: 0.7944903581267218
Precisão: 0.7454545454545455
Recall: 0.673134328358209
F1-Score: 0.7074509803921568


In [46]:
# Mostrar a matriz de confusão
conf_matrix = confusion_matrix(y_test, y_pred_vanilla)

fig = px.imshow(conf_matrix,
                labels=dict(x='Predição', y='Real', color='Contagem'),
                x=['Not Converted', 'Converted'],
                y=['Not Converted', 'Converted'],
                color_continuous_scale='Viridis')

fig.update_traces(text=conf_matrix, texttemplate="%{z}")
fig.update_layout(coloraxis_showscale=False)
fig.update_layout(title='Matriz de Confusão - Modelo Vanilla')

fig.show()

In [47]:
# Mostrar a matriz de confusão
conf_matrix = confusion_matrix(y_test, y_pred_blending)

fig = px.imshow(conf_matrix,
                labels=dict(x='Predição', y='Real', color='Contagem'),
                x=['Not Converted', 'Converted'],
                y=['Not Converted', 'Converted'],
                color_continuous_scale='Viridis')

fig.update_traces(text=conf_matrix, texttemplate="%{z}")
fig.update_layout(coloraxis_showscale=False)
fig.update_layout(title='Matriz de Confusão - Modelo Blending')

fig.show()

## Calcular Importancia

In [61]:
# Calcular a importância das variáveis considerando o Stacking Classifier

importances_vanilla = []

for estimador in stacking_model_vanilla.estimators_:
    # Modelos lineares possuem coeficientes
    if hasattr(estimador, 'coef_'):
        importances_vanilla.append(np.abs(estimador.coef_[0]))
        print(f'Coeficientes do modelo {type(estimador).__name__}')
    # Modelos baseados em árvores
    elif hasattr(estimador, 'feature_importances_'):
        importances_vanilla.append(np.abs(estimador.feature_importances_))
        print(f'Feature Importances do modelo {type(estimador).__name__}')
    # Caso não encontre coef e feature importances
    else:
        print(f'Não foi possível calcular a importância para {type(estimador).__name__}')

Coeficientes do modelo SGDClassifier
Coeficientes do modelo SVC
Feature Importances do modelo DecisionTreeClassifier


In [62]:
# Calcular a média das importâncias
importancia_media_vanilla = np.mean(importances_vanilla, axis=0)

In [63]:
# Obter os nomes das features
feature_names = (numeric_features.tolist() +
                preprocessor.named_transformers_['cat']
                .get_feature_names_out(categorical_features).tolist())

In [64]:
# Criar um dataframe com nomes e importância
df_feature_importances_vanilla = pd.DataFrame({'Feature': feature_names, 'Importance': importancia_media_vanilla})


In [65]:
# Ordenar o Dataframe
df_feature_importances_vanilla = df_feature_importances_vanilla.sort_values(by='Importance', ascending=True)

In [66]:
# Mostrar o ranking de importância

fig_vanilla = px.bar(df_feature_importances_vanilla,
             x='Importance',
             y='Feature',
             orientation='h',
             title='Importância das features dos algortimos base vanilla'
)

fig_vanilla.update_layout(height=1280, width=1000)

fig_vanilla.show()

OBS: Os valores de importância foram extraidos dos modelos base, portanto o resultado para o vanilla e blending é mesmo nesse caso